In [1]:
import os
import random
import shutil
import kagglehub

from pathlib import Path
from sklearn.model_selection import train_test_split

C:\Users\Naveen Singh Rawat\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_path = kagglehub.dataset_download(
    "zaidpy/oral-cancer-dataset"
)

print("Source dataset:", dataset_path)

Source dataset: C:\Users\Naveen Singh Rawat\.cache\kagglehub\datasets\zaidpy\oral-cancer-dataset\versions\2


In [3]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

class_folders = {}

for root, dirs, files in os.walk(dataset_path):
    valid_images = [
        file for file in files
        if file.lower().endswith(IMAGE_EXTENSIONS)
    ]

    if valid_images:
        class_name = os.path.basename(root)
        class_folders[class_name] = root

print("Detected classes:")

for class_name, folder in class_folders.items():
    print(class_name, "->", folder)

Detected classes:
CANCER -> C:\Users\Naveen Singh Rawat\.cache\kagglehub\datasets\zaidpy\oral-cancer-dataset\versions\2\Oral cancer Dataset 2.0\OC Dataset kaggle new\CANCER
NON CANCER -> C:\Users\Naveen Singh Rawat\.cache\kagglehub\datasets\zaidpy\oral-cancer-dataset\versions\2\Oral cancer Dataset 2.0\OC Dataset kaggle new\NON CANCER


In [4]:
project_root = Path.cwd().parent

output_root = project_root / "dataset" / "processed"

print("Project root:", project_root)
print("Output folder:", output_root)

Project root: e:\PROJECTS\OralScan-AI
Output folder: e:\PROJECTS\OralScan-AI\dataset\processed


In [5]:
if output_root.exists():
    shutil.rmtree(output_root)

output_root.mkdir(parents=True, exist_ok=True)

print("Processed dataset folder created.")

Processed dataset folder created.


In [6]:
splits = ["train", "validation", "test"]

for split in splits:
    for class_name in class_folders:
        folder = output_root / split / class_name
        folder.mkdir(parents=True, exist_ok=True)

print("Folder structure created.")

Folder structure created.


In [7]:
random.seed(42)

split_summary = {}

for class_name, source_folder in class_folders.items():

    image_files = [
        file for file in os.listdir(source_folder)
        if file.lower().endswith(IMAGE_EXTENSIONS)
    ]

    train_files, temporary_files = train_test_split(
        image_files,
        test_size=0.20,
        random_state=42,
        shuffle=True
    )

    validation_files, test_files = train_test_split(
        temporary_files,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    split_data = {
        "train": train_files,
        "validation": validation_files,
        "test": test_files
    }

    split_summary[class_name] = {
        split: len(files)
        for split, files in split_data.items()
    }

    for split, files in split_data.items():

        destination_folder = output_root / split / class_name

        for file_name in files:
            source_file = Path(source_folder) / file_name
            destination_file = destination_folder / file_name

            shutil.copy2(source_file, destination_file)

print("Dataset split completed.")

Dataset split completed.


In [8]:
for class_name, counts in split_summary.items():

    print(f"\nClass: {class_name}")

    for split, count in counts.items():
        print(f"{split}: {count}")


Class: CANCER
train: 400
validation: 50
test: 50

Class: NON CANCER
train: 360
validation: 45
test: 45


In [9]:
for split in splits:

    print(f"\n{split.upper()}")

    for class_name in class_folders:

        folder = output_root / split / class_name

        count = len([
            file for file in folder.iterdir()
            if file.suffix.lower() in IMAGE_EXTENSIONS
        ])

        print(f"{class_name}: {count}")


TRAIN
CANCER: 400
NON CANCER: 360

VALIDATION
CANCER: 50
NON CANCER: 45

TEST
CANCER: 50
NON CANCER: 45


In [10]:
for class_name in class_folders:

    train_names = {
        file.name
        for file in (output_root / "train" / class_name).iterdir()
    }

    validation_names = {
        file.name
        for file in (output_root / "validation" / class_name).iterdir()
    }

    test_names = {
        file.name
        for file in (output_root / "test" / class_name).iterdir()
    }

    print(f"\nClass: {class_name}")
    print(
        "Train ∩ Validation:",
        len(train_names.intersection(validation_names))
    )
    print(
        "Train ∩ Test:",
        len(train_names.intersection(test_names))
    )
    print(
        "Validation ∩ Test:",
        len(validation_names.intersection(test_names))
    )


Class: CANCER
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

Class: NON CANCER
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
